# Pulling the data

### 1. Pick 6–8 signals max

### 2. Make each one represent a different force:
- Growth
- Inflation
- Fear
- Energy
- Liquidity
- Future demand

### 3. Force each variable to justify its existence:
- If two say the same thing → kill one


### 4. Then build a composite index
- Z-score standardisation
- Weighted (not equal… think carefully)

#### Instead of tracking the dollar directly, track its impact:

Copper falling + dollar rising → global slowdown signal
Gold rising + dollar rising → fear spike (unusual, powerful)
Oil rising + dollar rising → inflation pressure intensifying

Now your system isn’t just descriptive.

It becomes predictive tension.

#### Fields to add

- rents
- mortgage rates
- food inflation
- vacancies
- real disposable income, if feasible

In [3]:
# UK Economic Pulse - Building the Core Logic Step by Step

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred
import yfinance as yf
import openpyxl


# Need pandas to connect to dataframes
# Matplotlib is sufficient for simple visualisations
# Fred will give us the data

import io
import re
import requests
import lxml


In [4]:
# FRED_API_KEY = os.getenv("FRED_API_KEY")  # or paste your key as a string for now
FRED_API_KEY = '0acae73adc29df31d82dce5320829f70'
fred = Fred(api_key=FRED_API_KEY)

# Why is it cleaner to use an environment variable?

In [5]:
# Get GDP, House Prices and Asset Prices from FRED

gdp = fred.get_series("NGDPRSAXDCGBQ")
gdp_us = fred.get_series("GDPC1")
house_prices = fred.get_series("QGBR628BIS")

# Commodity Prices — Gold, Silver, Copper, Natural Gas
# All sourced from COMEX/NYMEX futures via yfinance, daily

gold    = yf.download('GC=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/troy oz
silver  = yf.download('SI=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/troy oz
copper  = yf.download('HG=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/lb
nat_gas = yf.download('NG=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/MMBtu

# In many regions, Gas sets the marginal price of electricity.

# Market Indicators — Semiconductors, Uranium, US Dollar
# SOX & DXY: yfinance (daily)
# Trade-Weighted USD: FRED DTWEXBGS (daily, Federal Reserve)

sox     = yf.download('^SOX',     start='2000-01-01', progress=False)['Close'].squeeze()  # PHLX Semiconductor Index
uranium = yf.download('URA',      start='2009-01-01', progress=False)['Close'].squeeze()  # Global X Uranium ETF
dxy     = yf.download('DX-Y.NYB', start='2000-01-01', progress=False)['Close'].squeeze()  # ICE US Dollar Index
twusd   = fred.get_series('DTWEXBGS')                                                      # Nominal Broad Trade-Weighted USD

for s in [gold, silver, copper, nat_gas, sox, uranium, dxy]:
    s.index = pd.to_datetime(s.index)

In [6]:
# Get US Credit Spreads from FRED

series = {
    "us_hy_spread": "BAMLH0A0HYM2",
    "us_ig_spread": "BAMLC0A0CM",
    "us_bbb_spread": "BAMLC0A4CBBB"
}

us_credit_spreads = pd.DataFrame({name: fred.get_series(code) for name, code in series.items()})

In [7]:
# Get Bond Yields from FRED

def get_bond_yields() -> pd.DataFrame:
    series_map = {
        # US Treasuries (daily)
        "US_2Y": "DGS2",
        "US_10Y": "DGS10",
        "US_30Y": "DGS30",
        
        # UK Gilts (monthly via OECD on FRED)
        "UK_10Y": "IRLTLT01GBM156N",
        "UK_3M": "IR3TIB01GBM156N"
    }

    data = {}

    for name, series_id in series_map.items():
        s = fred.get_series(series_id)
        s.name = name
        data[name] = s

    df = pd.concat(data.values(), axis=1)

    # Sort and clean
    df = df.sort_index()

    # Align frequencies → monthly (critical)
    df = df.resample("ME").mean()

    return df

In [8]:
# -----------------------------
# 4) ONS Retail Sales Volume Index
# Sheet: KPSA (Seasonally Adjusted), headers at row 7
# -----------------------------

def get_uk_retail_sales_ons() -> pd.Series:
    url = "https://www.ons.gov.uk/file?uri=/businessindustryandtrade/retailindustry/datasets/retailsalesindexreferencetables/current/mainreferencetables.xlsx"

    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_excel(io.BytesIO(r.content), sheet_name="KPSA", header=6)

    time_col = "Time Period"
    value_col = "All retailing including automotive fuel [note1]"
    df = df[[time_col, value_col]].copy()

    # Truncate before the 'Revision to index numbers' section
    revision_mask = df[time_col].astype(str).str.contains("Revision", case=False, na=False)
    if revision_mask.any():
        df = df.iloc[:revision_mask.values.argmax()]

    # Drop blank / metadata rows
    df = df[df[time_col].notna()]
    df = df[df[time_col].astype(str).str.strip().ne("")]

    # Parse 'yyyy-mmm' (e.g. '1996 Jan') to datetime
    date_str = df[time_col].astype(str).str.strip().str.replace("-", " ")
    df["date"] = pd.to_datetime(date_str, format="%Y %b", errors="coerce")

    df["value"] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=["date", "value"])

    # Sort chronologically on datetime BEFORE converting to string
    df = df.sort_values("date")
    return df.set_index("date")["value"]

In [9]:
# -----------------------------
# 1) ONS CPI (index level)
# Series: D7BT = CPI INDEX 00: ALL ITEMS 2015=100
# Source dataset: MM23
# -----------------------------

def get_uk_cpi_index_ons() -> pd.Series:
    url = "https://www.ons.gov.uk/generator?format=csv&uri=/economy/inflationandpriceindices/timeseries/d7bt/mm23"
    
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))

    # Rename the two columns into something sane
    df.columns = ["raw_date", "raw_value"]

    # Keep only monthly rows like '1988 JAN'
    monthly_mask = df["raw_date"].astype(str).str.match(r"^\d{4} [A-Z]{3}$")
    df = df.loc[monthly_mask].copy()

    # Rename
    df = df.rename(columns={
        "raw_date": "date",
        "raw_value": "cpi_index"
    })

    # Parse fields
    df["date"] = pd.to_datetime(df["date"], format="%Y %b", errors="coerce")
    df["cpi_index"] = pd.to_numeric(df["cpi_index"], errors="coerce")

    # Clean up
    df = df.dropna(subset=["date", "cpi_index"]).sort_values("date")
    df = df.set_index("date")

    s = df["cpi_index"]
    s.name = "cpi_index"
    return s

In [10]:
# -----------------------------
# 2) ONS Nominal Wages (AWE Total Pay)
# Series: KAB9
# Source dataset: LMS
# -----------------------------

def get_uk_nominal_wages_ons() -> pd.Series:
    url = "https://www.ons.gov.uk/generator?format=csv&uri=/employmentandlabourmarket/peopleinwork/earningsandworkinghours/timeseries/kab9/lms"
    
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))

    df.columns = ["raw_date", "raw_value"]

    # Monthly rows like '2000 JAN'
    monthly_mask = df["raw_date"].astype(str).str.match(r"^\d{4} [A-Z]{3}$")
    df = df.loc[monthly_mask].copy()

    df = df.rename(columns={
        "raw_date": "date",
        "raw_value": "nominal_wage"
    })

    df["date"] = pd.to_datetime(df["date"], format="%Y %b", errors="coerce")
    df["nominal_wage"] = pd.to_numeric(df["nominal_wage"], errors="coerce")

    df = df.dropna(subset=["date", "nominal_wage"]).sort_values("date")
    df = df.set_index("date")

    s = df["nominal_wage"]
    s.name = "nominal_wage"
    return s

def get_uk_real_wages(cpi_index: pd.Series, wages: pd.Series) -> pd.Series:
    # Align dates
    df = pd.concat([cpi_index, wages], axis=1).dropna()

    # Real wage index (base-adjusted)
    df["real_wage"] = df["nominal_wage"] / df["cpi_index"] * 100

    s = df["real_wage"]
    s.name = "real_wage"
    return s

In [11]:
def _parse_ons_monthly_series_from_csv(url: str, series_name: str) -> pd.Series:
    """
    Helper to pull an ONS timeseries CSV and return a clean monthly pandas Series.

    The ONS CSV usually comes with 2 columns:
    - raw_date like '1988 JAN' or '2025 NOV-JAN'
    - raw_value

    We keep only monthly rows like '1988 JAN' and ignore rolling-quarter labels
    like '2025 NOV-JAN' if they appear.
    """
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))
    df.columns = ["raw_date", "raw_value"]

    # Keep rows like '1988 JAN'
    monthly_mask = df["raw_date"].astype(str).str.match(r"^\d{4}\s[A-Z]{3}$", na=False)
    df = df.loc[monthly_mask].copy()

    # Clean numeric values
    df["raw_value"] = pd.to_numeric(df["raw_value"], errors="coerce")
    df = df.dropna(subset=["raw_value"])

    # Parse dates
    df["date"] = pd.to_datetime(df["raw_date"], format="%Y %b")

    s = (
        df.set_index("date")["raw_value"]
        .sort_index()
        .rename(series_name)
    )

    return s

def get_uk_unemployment_rate_ons() -> pd.Series:
    """
    UK unemployment rate, aged 16 and over, seasonally adjusted (%).

    ONS series ID: MGSX
    Source dataset: Labour market statistics (LMS)
    """
    url = (
        "https://www.ons.gov.uk/generator?format=csv&uri="
        "/employmentandlabourmarket/peoplenotinwork/unemployment/timeseries/mgsx/lms"
    )
    return _parse_ons_monthly_series_from_csv(url, "uk_unemployment_rate")

def get_uk_labour_participation_rate_ons() -> pd.Series:
    """
    UK labour market participation rate, proxied as:
        100 - economic inactivity rate

    Uses ONS economic inactivity rate:
    UK, all, aged 16-64, seasonally adjusted (%)

    ONS series ID: LF2S
    """
    url = (
        "https://www.ons.gov.uk/generator?format=csv&uri="
        "/employmentandlabourmarket/peoplenotinwork/economicinactivity/timeseries/lf2s/lms"
    )

    inactivity = _parse_ons_monthly_series_from_csv(url, "uk_economic_inactivity_rate")
    participation = (100 - inactivity).rename("uk_labour_participation_rate")

    return participation

In [12]:
# -----------------------------
# 3) Bank of England policy rate
# Pull from the official Bank Rate history page
# -----------------------------

def get_boe_policy_rate(local_fallback_path=None) -> pd.Series:
    url = "https://www.bankofengland.co.uk/boeapps/database/Bank-Rate.asp"

    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-GB,en;q=0.9",
        "Referer": "https://www.bankofengland.co.uk/boeapps/database/",
        "Connection": "keep-alive",
    })

    # First hit the database home page to look less like a bot
    try:
        warmup = session.get(
            "https://www.bankofengland.co.uk/boeapps/database/",
            timeout=30
        )
        warmup.raise_for_status()
        time.sleep(1.0)
    except Exception:
        pass

    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()

        tables = pd.read_html(io.StringIO(r.text))
        if not tables:
            raise ValueError("No tables found on Bank Rate page.")

        df = tables[0].copy()
        df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

        if "date_changed" not in df.columns or "rate" not in df.columns:
            raise ValueError(f"Unexpected columns on Bank Rate page: {list(df.columns)}")

        df["date_changed"] = pd.to_datetime(df["date_changed"], format="%d %b %y", errors="coerce")
        df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

        df = df.dropna(subset=["date_changed", "rate"]).sort_values("date_changed")
        df = df.set_index("date_changed")

        s = df["rate"]
        s.name = "policy_rate"
        return s

    except requests.HTTPError as e:
        if local_fallback_path is None:
            raise RuntimeError(
                f"BoE request blocked with HTTP {e.response.status_code}. "
                "Download the Bank Rate history CSV manually from the BoE page and pass its path "
                "as local_fallback_path."
            ) from e

        # Fallback: read a manually downloaded BoE CSV
        raw = pd.read_csv(local_fallback_path)

        # Adjust if your CSV comes with slightly different column names
        raw.columns = [str(c).strip().lower().replace(" ", "_") for c in raw.columns]

        # Try common variants
        date_col = next((c for c in raw.columns if c in ["date_changed", "date"]), None)
        rate_col = next((c for c in raw.columns if c in ["rate", "value"]), None)

        if date_col is None or rate_col is None:
            raise ValueError(
                f"Could not identify date/rate columns in fallback CSV: {list(raw.columns)}"
            )

        df = raw[[date_col, rate_col]].copy()
        df.columns = ["date_changed", "rate"]

        df["date_changed"] = pd.to_datetime(df["date_changed"], errors="coerce", dayfirst=True)
        df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

        df = df.dropna(subset=["date_changed", "rate"]).sort_values("date_changed")
        df = df.set_index("date_changed")

        s = df["rate"]
        s.name = "policy_rate"
        return s

# ----------------------------- 
# 4) Convert policy rate changes to month-end series 
# -----------------------------

def make_policy_rate_monthly(policy_rate_changes: pd.Series, end_date=None) -> pd.Series:
    monthly = policy_rate_changes.resample("ME").last()

    if end_date is None:
        end_date = pd.Timestamp.today().to_period("M").to_timestamp("M")

    full_index = pd.date_range(start=monthly.index.min(), end=end_date, freq="ME")

    monthly = monthly.reindex(full_index).ffill()
    monthly.name = "policy_rate"
    return monthly

In [13]:
# -----------------------------
# 5) Call the functions
# -----------------------------

# Economic Activity

retail_sales = get_uk_retail_sales_ons()

# Inflation Pressure

cpi_index = get_uk_cpi_index_ons()
wages = get_uk_nominal_wages_ons()
real_wages = get_uk_real_wages(cpi_index, wages)
uk_unemployment_rate = get_uk_unemployment_rate_ons()
uk_labour_participation_rate = get_uk_labour_participation_rate_ons()

# Financial Conditions

policy_rate_changes = get_boe_policy_rate()
policy_rate_monthly = make_policy_rate_monthly(policy_rate_changes)
bond_yields = get_bond_yields() # df
us_credit_spreads = pd.DataFrame({name: fred.get_series(code) for name, code in series.items()}) # df

# Asset prices

oil = fred.get_series("DCOILBRENTEU")
# etc 



In [14]:
# Inspect the frequency

for name, s in {

    # Economic Activity
    "GDP (UK)":           gdp,
    "GDP (US)":           gdp_us,
    "House Prices (UK)":  house_prices,
    "Retail Sales (UK)":  retail_sales,

    # Inflation Pressure
    "CPI (UK)":           cpi_index,
    "Real Wages (UK)":    real_wages,

    # Financial Conditions
    "Base Rate (UK)":             policy_rate_monthly,
    "Bond Yields (US 2Y)":        bond_yields["US_2Y"],
    "Bond Yields (US 10Y)":       bond_yields["US_10Y"],
    "Bond Yields (US 30Y)":       bond_yields["US_30Y"],
    "Bond Yields (UK 10Y)":       bond_yields["UK_10Y"],
    "Bond Yields (UK 3M)":        bond_yields["UK_3M"],
    "US High Yield Spread":       us_credit_spreads["us_hy_spread"],
    "US IG Spread":               us_credit_spreads["us_ig_spread"],
    "US BBB Spread":              us_credit_spreads["us_bbb_spread"],

    # Labour Market
    "Unemployment Rate (UK)":         uk_unemployment_rate,
    "Labour Participation Rate (UK)": uk_labour_participation_rate,

    # Commodity Prices
    "Oil":                oil,
    "Gold":               gold,
    "Silver":             silver,
    "Copper":             copper,
    "Natural Gas":        nat_gas,

    # Market Indicators
    "SOX (Semiconductors)":       sox,
    "Uranium (URA ETF)":          uranium,
    "DXY (US Dollar Index)":      dxy,
    "Trade-Weighted USD":         twusd

}.items():
    print(name)
    print("Start:", s.index.min())
    print("End:  ", s.index.max())
    print("Missing:", s.isna().sum())
    print("Inferred freq:", pd.infer_freq(s.index[:20]) if len(s) > 20 else pd.infer_freq(s.index))
    print("-" * 40)

GDP (UK)
Start: 1955-01-01 00:00:00
End:   2025-10-01 00:00:00
Missing: 0
Inferred freq: QS-OCT
----------------------------------------
GDP (US)
Start: 1947-01-01 00:00:00
End:   2026-01-01 00:00:00
Missing: 0
Inferred freq: QS-OCT
----------------------------------------
House Prices (UK)
Start: 1968-04-01 00:00:00
End:   2025-10-01 00:00:00
Missing: 0
Inferred freq: QS-OCT
----------------------------------------
Retail Sales (UK)
Start: 1996-01-01 00:00:00
End:   2026-03-01 00:00:00
Missing: 0
Inferred freq: MS
----------------------------------------
CPI (UK)
Start: 1988-01-01 00:00:00
End:   2026-03-01 00:00:00
Missing: 0
Inferred freq: MS
----------------------------------------
Real Wages (UK)
Start: 2000-01-01 00:00:00
End:   2026-02-01 00:00:00
Missing: 0
Inferred freq: MS
----------------------------------------
Base Rate (UK)
Start: 1975-01-31 00:00:00
End:   2026-05-31 00:00:00
Missing: 0
Inferred freq: ME
----------------------------------------
Bond Yields (US 2Y)
Start:

In [15]:
# Economic Activity
gdp_monthly          = gdp.resample('ME').ffill()
gdp_us_monthly       = gdp_us.resample('ME').ffill()
house_prices_monthly = house_prices.resample('ME').ffill()
retail_sales_monthly = retail_sales.resample('ME').last()

# Labour Market
uk_unemployment_rate_monthly        = uk_unemployment_rate.resample('ME').last()        # already monthly, % level
uk_labour_participation_rate_monthly = uk_labour_participation_rate.resample('ME').last() # already monthly, % level

# Inflation Pressure
cpi_monthly          = cpi_index.resample('ME').last()
real_wages_monthly   = real_wages.resample('ME').last()

# Labour Market
uk_unemployment_rate_monthly        = uk_unemployment_rate.resample('ME').last()   # already monthly, % level
uk_labour_participation_rate_monthly = uk_labour_participation_rate.resample('ME').last()  # already monthly, % level

# Financial Conditions
policy_rate_monthly        = policy_rate_monthly.copy()
bond_yields_monthly        = bond_yields.resample('ME').mean()        # df: US_2Y, US_10Y, US_30Y, UK_10Y, UK_3M
us_credit_spreads_monthly  = us_credit_spreads.resample('ME').mean()  # df: us_hy_spread, us_ig_spread, us_bbb_spread

# Commodity Prices
oil_monthly          = oil.resample('ME').mean()
gold_monthly         = gold.resample('ME').mean()
silver_monthly       = silver.resample('ME').last()
copper_monthly       = copper.resample('ME').last()
nat_gas_monthly      = nat_gas.resample('ME').mean()

# Market Indicators
sox_monthly          = sox.resample('ME').mean()
uranium_monthly      = uranium.resample('ME').mean()
dxy_monthly          = dxy.resample('ME').mean()
twusd_monthly        = twusd.resample('ME').mean()

# ffill()  → GDP is quarterly, carry forward until next reading
# mean()   → daily series: average over the month
# last()   → monthly point-in-time series: take end-of-month value
# copy()   → already month-end, no resampling needed

print(gdp_monthly.tail())

2025-06-30    705117.0
2025-07-31    705681.0
2025-08-31    705681.0
2025-09-30    705681.0
2025-10-31    706067.0
Freq: ME, dtype: float64


In [25]:
absolute_values = pd.DataFrame({

    # Economic Activity
    "gdp":            gdp_monthly,
    "gdp_us":         gdp_us_monthly,
    "house_prices":   house_prices_monthly,
    "retail_sales":   retail_sales_monthly,

    # Labour Market
    "unemployment_rate":   uk_unemployment_rate_monthly,        # %, e.g. 5.0 = 5%
    "participation_rate":  uk_labour_participation_rate_monthly, # %, e.g. 63.0 = 63%

    # Inflation Pressure
    "cpi":            cpi_monthly,
    "real_wages":     real_wages_monthly,

    # Financial Conditions
    "policy_rate":    policy_rate_monthly,
    "yield_us_2y":    bond_yields_monthly["US_2Y"],
    "yield_us_10y":   bond_yields_monthly["US_10Y"],
    "yield_us_30y":   bond_yields_monthly["US_30Y"],
    "yield_uk_10y":   bond_yields_monthly["UK_10Y"],
    "yield_uk_3m":    bond_yields_monthly["UK_3M"],
    "spread_us_hy":   us_credit_spreads_monthly["us_hy_spread"],
    "spread_us_ig":   us_credit_spreads_monthly["us_ig_spread"],
    "spread_us_bbb":  us_credit_spreads_monthly["us_bbb_spread"],

    # Commodity Prices
    "oil":            oil_monthly,
    "gold":           gold_monthly,
    "silver":         silver_monthly,
    "copper":         copper_monthly,
    "nat_gas":        nat_gas_monthly,

    # Market Indicators
    "sox":            sox_monthly,
    "uranium":        uranium_monthly,
    "dxy":            dxy_monthly,
    "twusd":          twusd_monthly,

})

absolute_values.tail(5).T

,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31
gdp,NaN,NaN,NaN,NaN,NaN
gdp_us,24174.527000,NaN,NaN,NaN,NaN
house_prices,NaN,NaN,NaN,NaN,NaN
retail_sales,103.700000,103.000000,103.700000,NaN,NaN
unemployment_rate,4.900000,NaN,NaN,NaN,NaN
participation_rate,79.000000,NaN,NaN,NaN,NaN
cpi,139.500000,140.100000,141.000000,NaN,NaN
real_wages,532.616487,531.763026,NaN,NaN,NaN
policy_rate,3.750000,3.750000,3.750000,3.750000,3.750000
yield_us_2y,3.537000,3.471579,3.714545,3.799091,3.925000


In [17]:
absolute_values.to_csv('/Users/ornettematthews/Documents/GitHub/uk-economic-pulse/data/raw/absolute_values.csv')


In [20]:
signals = pd.DataFrame(index=absolute_values.index)

# --- Economic Activity ---
# YoY % change captures growth momentum vs. same period last year
signals["gdp_growth_yoy"]    = absolute_values["gdp"].pct_change(12, fill_method=None) * 100
signals["gdp_growth_yoy_us"] = absolute_values["gdp_us"].pct_change(12, fill_method=None) * 100
signals["house_prices_yoy"]  = absolute_values["house_prices"].pct_change(12, fill_method=None) * 100
signals["retail_sales_yoy"]  = absolute_values["retail_sales"].pct_change(12, fill_method=None) * 100

# --- Inflation Pressure ---
signals["cpi_yoy"]         = absolute_values["cpi"].pct_change(12, fill_method=None) * 100
signals["real_wages_yoy"]  = absolute_values["real_wages"].pct_change(12, fill_method=None) * 100

# --- Labour Market ---
# Levels: unemployment and participation are already meaningful as % readings
signals["unemployment_rate"]  = absolute_values["unemployment_rate"]
signals["participation_rate"] = absolute_values["participation_rate"]

# --- Financial Conditions ---
# Rates and spreads are already in level form — no transformation needed
signals["policy_rate"]     = absolute_values["policy_rate"]
signals["yield_us_2y"]     = absolute_values["yield_us_2y"]
signals["yield_us_10y"]    = absolute_values["yield_us_10y"]
signals["yield_us_30y"]    = absolute_values["yield_us_30y"]
signals["yield_uk_10y"]    = absolute_values["yield_uk_10y"]
signals["yield_uk_3m"]     = absolute_values["yield_uk_3m"]
signals["spread_us_hy"]    = absolute_values["spread_us_hy"]
signals["spread_us_ig"]    = absolute_values["spread_us_ig"]
signals["spread_us_bbb"]   = absolute_values["spread_us_bbb"]

# Derived: yield curve shape and cross-country divergence
# Negative = inverted curve = historically precedes recession
signals["us_yield_curve"]     = absolute_values["yield_us_10y"] - absolute_values["yield_us_2y"]
signals["uk_yield_curve"]     = absolute_values["yield_uk_10y"] - absolute_values["yield_uk_3m"]
signals["uk_us_10y_spread"]   = absolute_values["yield_uk_10y"] - absolute_values["yield_us_10y"]

# --- Commodity Prices ---
# YoY % change captures price pressure and demand signals
signals["oil_yoy"]     = absolute_values["oil"].pct_change(12, fill_method=None) * 100
signals["gold_yoy"]    = absolute_values["gold"].pct_change(12, fill_method=None) * 100
signals["silver_yoy"]  = absolute_values["silver"].pct_change(12, fill_method=None) * 100
signals["copper_yoy"]  = absolute_values["copper"].pct_change(12, fill_method=None) * 100  # "Dr. Copper" — global demand barometer
signals["nat_gas_yoy"] = absolute_values["nat_gas"].pct_change(12, fill_method=None) * 100

# --- Market Indicators ---
# Equity indices and USD: YoY % change captures trend and momentum
signals["sox_yoy"]     = absolute_values["sox"].pct_change(12, fill_method=None) * 100
signals["uranium_yoy"] = absolute_values["uranium"].pct_change(12, fill_method=None) * 100
signals["dxy_yoy"]     = absolute_values["dxy"].pct_change(12, fill_method=None) * 100
signals["twusd_yoy"]   = absolute_values["twusd"].pct_change(12, fill_method=None) * 100

signals.tail(10).T

,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31
gdp_growth_yoy,1.281518,1.281518,0.997434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gdp_growth_yoy_us,2.335168,2.335168,1.989300,1.989300,1.989300,2.659722,NaN,NaN,NaN,NaN
house_prices_yoy,-1.283979,-1.283979,-0.881640,NaN,NaN,NaN,NaN,NaN,NaN,NaN
retail_sales_yoy,0.892857,1.984127,1.896208,1.698302,1.596806,4.641776,1.879327,1.666667,NaN,NaN
cpi_yoy,3.723008,3.800298,3.555556,3.256847,3.318584,3.028065,3.014706,3.296703,NaN,NaN
real_wages_yoy,1.390226,0.444185,1.360606,1.366266,-0.365285,1.572215,0.583827,NaN,NaN,NaN
unemployment_rate,5.000000,5.100000,5.100000,5.200000,5.200000,4.900000,NaN,NaN,NaN,NaN
participation_rate,79.000000,79.000000,79.200000,79.200000,79.300000,79.000000,NaN,NaN,NaN,NaN
policy_rate,4.000000,4.000000,4.000000,4.000000,3.750000,3.750000,3.750000,3.750000,3.750000,3.750000
yield_us_2y,3.703810,3.568571,3.521364,3.550000,3.500909,3.537000,3.471579,3.714545,3.799091,3.925000


In [21]:
signals.to_csv('/Users/ornettematthews/Documents/GitHub/uk-economic-pulse/data/raw/signals.csv')